# 09 — Summarization → Zero-Shot Labeling

A different approach to auto-labeling: instead of clustering embeddings or
fine-tuning a classifier, summarize each article with a text-summarization
model, then zero-shot-classify the summary into one of the 4 AG News
classes. Uses `utils.config.SUMMARIZATION_MODEL_NAME`
(`sshleifer/distilbart-cnn-6-6`) and `utils.config.ZERO_SHOT_MODEL_NAME`
(`valhalla/distilbart-mnli-12-3`) — both open-source, CPU-runnable, no
fine-tuning. Also generates a short headline per article (same summarizer,
shorter max_length) alongside its summary, saved together in the
sample-labels CSV. Scored against the hidden true label like every other
method in `results/comparison_table.csv`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.metrics import evaluate_semisupervised
from utils.samples import save_label_samples
from utils.summarization import generate_titles, summarize_texts, zero_shot_label

In [2]:
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")
test_sample = stratified_sample(test_clean, config.SUMMARIZATION_SAMPLE_SIZE, seed=config.SEED)
print(f"Summarizing + zero-shot labeling {len(test_sample)} test rows")

Summarizing + zero-shot labeling 200 test rows


In [3]:
summaries = summarize_texts(test_sample["text"].tolist())
generated_titles = generate_titles(test_sample["text"].tolist())
predicted_labels, confidence = zero_shot_label(summaries, config.CLASS_NAMES)

print("Example:")
print(" text:", test_sample["text"].iloc[0][:100])
print(" summary:", summaries[0])
print(" generated title:", generated_titles[0])
print(" predicted:", config.CLASS_NAMES[predicted_labels[0]], f"(confidence {confidence[0]:.2f})")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/262 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

Example:
 text: Large Explosion Heard in Central Baghdad (Reuters) Reuters - A large blast was heard in central\Bagh
 summary: Large blast heard in central Baghdad on Thursday, witnesses said .
 generated title: Large blast heard in central Baghdad on Thursday .
 predicted: World (confidence 0.48)


In [4]:
results, report, cm = evaluate_semisupervised(
    test_sample["label"].to_numpy(), predicted_labels, config.CLASS_NAMES,
    save_path=config.RESULTS_DIR / "confusion_matrix_summarization.png")
print(report)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_summarization.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved summarization + zero-shot labeling results.")

              precision    recall  f1-score   support

       World       0.59      0.84      0.69        50
      Sports       0.96      0.88      0.92        50
    Business       0.57      0.80      0.67        50
    Sci/Tech       0.69      0.18      0.29        50

    accuracy                           0.68       200
   macro avg       0.70      0.68      0.64       200
weighted avg       0.70      0.68      0.64       200

Saved summarization + zero-shot labeling results.


In [5]:
save_label_samples(
    test_sample["text"], predicted_labels, test_sample["label"].to_numpy(),
    config.CLASS_NAMES, confidence=confidence, n_per_class=2, seed=config.SEED,
    extra_columns={"summary": summaries, "generated_title": generated_titles},
    path=config.RESULTS_DIR / "sample_labels_summarization.csv")
print("Saved sample generated labels (with summaries + generated titles) for summarization_zero_shot.")

Saved sample generated labels (with summaries + generated titles) for summarization_zero_shot.
